#03_gold_mart_member_month.sql

Gold - Mart Beneficiario x Competencia

Este notebook consolida em uma única tabela mensal (beneficiario x competencia):

- financeiro (faturas)
- assistencial (eventos)
- experiência (SAC)
- digital (app logs)

##Objetivo

- entrega uma tabela pronta para BI e análises (churn, sinistralidade, experiencia)
- padroniza granulidade mensal

##Estratégia
- full refresh (insert overwrite) para validação inicial
- CTEs (WITH) para robustez (sem TEMP VIEW)

##Contexto

In [0]:
USE CATALOG healthcare_dev;

##Cria tabela mart_member_month

In [0]:
CREATE TABLE IF NOT EXISTS healthcare_dev.gold.mart_member_month (
  beneficiario_id STRING,
  competencia STRING,

  --dimensões
  sexo STRING,
  uf STRING,
  municipio STRING,
  canal_aquisicao STRING,
  segmento_vinculo STRING,

  --financeiro
  valor_faturado DECIMAL(18,2),
  valor_pago DECIMAL(18,2),
  status_pagamento STRING,
  flag_inadimpencia INT,

  -- assistencial
  qtd_eventos BIGINT,
  custo_total DECIMAL(18,2),
  qtd_eventos_custo_neg BIGINT,
  custo_neg_total DECIMAL(18,2),
  pct_eventos_custo_neg DOUBLE,

  -- experiência (SAC)
  qtd_protocolos BIGINT,
  nps_medio DOUBLE,
  csat_medio DOUBLE,
  qtd_sla_inconsistente BIGINT,

  -- digital (app)
  qtd_eventos_app BIGINT,
  qtd_http_invalid BIGINT,
  latencia_media DOUBLE,

  gold_build_ts TIMESTAMP
)
USING DELTA;

##Constroe dataset gold

In [0]:
INSERT OVERWRITE healthcare_dev.gold.mart_member_month
WITH
mm_base AS (
  SELECT DISTINCT beneficiario_id, competencia
  FROM healthcare_dev.silver.fact_faturas
),
mm_dim AS (
  SELECT beneficiario_id, sexo, uf, municipio, canal_aquisicao, segmento_vinculo
  FROM healthcare_dev.silver.dim_beneficiarios
),
mm_fin AS (
  SELECT
    beneficiario_id,
    competencia,
    SUM(valor_faturado) AS valor_faturado,
    SUM(COALESCE(valor_pago, 0)) AS valor_pago,
    MAX(status_pagamento) AS status_pagamento,
    CASE
      WHEN MAX(status_pagamento) IN ('ATRASADO','INADIMPLENTE') THEN 1
      WHEN SUM(COALESCE(valor_pago,0)) < SUM(valor_faturado) THEN 1
      ELSE 0
    END AS flag_inadimplencia
  FROM healthcare_dev.silver.fact_faturas
  GROUP BY beneficiario_id, competencia
),
mm_evt AS (
  SELECT
    beneficiario_id,
    date_format(data_evento, 'yyyy-MM') AS competencia,
    COUNT(*) AS qtd_eventos,
    SUM(COALESCE(valor_pago,0)) AS custo_total,
    SUM(CASE WHEN COALESCE(valor_pago,0) < 0 THEN 1 ELSE 0 END) AS qtd_eventos_custo_neg,
    SUM(CASE WHEN COALESCE(valor_pago,0) < 0 THEN COALESCE(valor_pago,0) ELSE 0 END) AS custo_neg_total
  FROM healthcare_dev.silver.fact_eventos
  GROUP BY beneficiario_id, date_format(data_evento, 'yyyy-MM')
),
mm_sac AS (
  SELECT
    beneficiario_id,
    date_format(data_abertura, 'yyyy-MM') AS competencia,
    COUNT(*) AS qtd_protocolos,
    AVG(CAST(nps AS DOUBLE)) AS nps_medio,
    AVG(CAST(csat AS DOUBLE)) AS csat_medio,
    SUM(CASE WHEN flag_tempo_resolucao_inconsistente = 1 THEN 1 ELSE 0 END) AS qtd_sla_inconsistente
  FROM healthcare_dev.silver.fact_sac
  GROUP BY beneficiario_id, date_format(data_abertura, 'yyyy-MM')
),
mm_app AS (
  SELECT
    beneficiario_id,
    date_format(timestamp_evento, 'yyyy-MM') AS competencia,
    COUNT(*) AS qtd_eventos_app,
    SUM(CASE WHEN flag_invalid_http_status = 1 THEN 1 ELSE 0 END) AS qtd_http_invalid,
    AVG(CAST(latencia_ms AS DOUBLE)) AS latencia_media
  FROM healthcare_dev.silver.fact_app_logs
  GROUP BY beneficiario_id, date_format(timestamp_evento, 'yyyy-MM')
)
SELECT
  b.beneficiario_id,
  b.competencia,

  d.sexo, d.uf, d.municipio, d.canal_aquisicao, d.segmento_vinculo,

  f.valor_faturado,
  f.valor_pago,
  f.status_pagamento,
  f.flag_inadimplencia,

  e.qtd_eventos,
  e.custo_total,
  e.qtd_eventos_custo_neg,
  e.custo_neg_total,
  CASE
    WHEN e.qtd_eventos IS NULL OR e.qtd_eventos = 0 THEN 0.0
    ELSE ROUND(100.0 * e.qtd_eventos_custo_neg / e.qtd_eventos, 4)
  END AS pct_eventos_custo_neg,

  s.qtd_protocolos,
  s.nps_medio,
  s.csat_medio,
  s.qtd_sla_inconsistente,

  a.qtd_eventos_app,
  a.qtd_http_invalid,
  a.latencia_media,
  current_timestamp() AS gold_build_ts
FROM mm_base b
LEFT JOIN mm_dim d ON b.beneficiario_id = d.beneficiario_id
LEFT JOIN mm_fin f ON b.beneficiario_id = f.beneficiario_id AND b.competencia = f.competencia
LEFT JOIN mm_evt e ON b.beneficiario_id = e.beneficiario_id AND b.competencia = e.competencia
LEFT JOIN mm_sac s ON b.beneficiario_id = s.beneficiario_id AND b.competencia = s.competencia
LEFT JOIN mm_app a ON b.beneficiario_id = a.beneficiario_id AND b.competencia = a.competencia;

##Checa sanidade

In [0]:
SELECT COUNT(*) AS n_gold_member_month
FROM healthcare_dev.gold.mart_member_month;